In [67]:
# Shared data loading and plot styling.
import json
import re
from collections import Counter
from datetime import datetime
from dateutil.relativedelta import relativedelta
from itertools import cycle
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

In [68]:
# Load and process data from consolidated sound_analysis files.
from datetime import timedelta, date
import plotly.express as px

SOUND_ANALYSIS_DIR = Path('sound_analysis')

# Use today's date as reference for all time frame calculations
TODAY = datetime.today().date()

def parse_timestamp(name):
    match = re.search(r'(\d{8}_\d{6})', name)
    return match.group(1) if match else None

def parse_timestamp_datetime(name):
    ts = parse_timestamp(name)
    if not ts:
        return None
    try:
        return datetime.strptime(ts, '%Y%m%d_%H%M%S')
    except ValueError:
        return None

def to_float(value, default=np.nan):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(default)

def to_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return int(default)

def load_json_if_exists(path):
    if path.exists():
        try:
            return json.loads(path.read_text(encoding='utf-8-sig'))
        except (json.JSONDecodeError, UnicodeDecodeError):
            try:
                return json.loads(path.read_text(encoding='utf-8'))
            except (json.JSONDecodeError, UnicodeDecodeError):
                return {}
    return {}

def parse_created_at_datetime(item, fallback_name=''):
    created_at = str(item.get('created_at', '')).strip()
    if created_at:
        try:
            return datetime.fromisoformat(created_at)
        except ValueError:
            pass
    return parse_timestamp_datetime(fallback_name)

def make_recording_label(recording_id, recording_name=''):
    if recording_name:
        return str(recording_name).replace('_', ' ')
    base = re.sub(r'_\d{8}_\d{6}$', '', recording_id)
    ts = parse_timestamp(recording_id)
    if not ts:
        return base.replace('_', ' ')
    try:
        dt = datetime.strptime(ts, '%Y%m%d_%H%M%S')
    except ValueError:
        return base.replace('_', ' ')
    generic_prefixes = {'filler_analysis', 'speed', 'pitch', 'transcript'}
    if base.lower() in generic_prefixes:
        return dt.strftime('%H:%M')
    return base.replace('_', ' ')

analysis_files = sorted(SOUND_ANALYSIS_DIR.glob('*.json'))
records = []
filler_words_set = set()

for analysis_path in analysis_files:
    item = load_json_if_exists(analysis_path)
    if not item:
        continue

    recording_id = analysis_path.stem
    recording_dt = parse_created_at_datetime(item, fallback_name=recording_id)
    if not recording_dt:
        continue

    recording_name = str(item.get('recording_name', '')).strip()
    speed = item.get('speed', {}) if isinstance(item.get('speed'), dict) else {}
    filler = item.get('filler', {}) if isinstance(item.get('filler'), dict) else {}
    pitch = item.get('pitch', {}) if isinstance(item.get('pitch'), dict) else {}
    pitch_summary = pitch.get('summary', {}) if isinstance(pitch.get('summary'), dict) else {}

    filler_counts_raw = filler.get('filler_word_counts', {}) if isinstance(filler, dict) else {}
    filler_counts = {
        str(word): to_int(count)
        for word, count in filler_counts_raw.items()
        if isinstance(word, str) and to_int(count) > 0
    }
    filler_words_set.update(filler_counts.keys())

    total_filler_words = to_int(filler.get('total_filler_words', sum(filler_counts.values())), default=0)
    total_words = to_int(speed.get('total_words', filler.get('total_words', 1)), default=1)
    filler_percentage = to_float(filler.get('filler_percentage', 0.0), default=0.0)

    pitch_avg_variation = to_float(pitch_summary.get('mean_pitch_semitones', np.nan))
    pitch_min_variation = to_float(pitch_summary.get('min_pitch_semitones', np.nan))
    pitch_max_variation = to_float(pitch_summary.get('max_pitch_semitones', np.nan))
    wpm = to_float(speed.get('wpm', np.nan))

    recording_label = make_recording_label(recording_id, recording_name=recording_name)

    records.append({
        'recording_id': recording_id,
        'recording_label': recording_label,
        'recording_dt': recording_dt,
        'filler_counts': filler_counts,
        'total_filler_words': total_filler_words,
        'total_words': total_words,
        'filler_percentage': filler_percentage,
        'pitch_avg_variation': pitch_avg_variation,
        'pitch_min_variation': pitch_min_variation,
        'pitch_max_variation': pitch_max_variation,
        'wpm': wpm,
    })

records = sorted(records, key=lambda r: (r['recording_dt'], r['recording_id']))
filler_words = sorted(list(filler_words_set))

COLORS = {
    'sage': '#C8E6C9',
    'seafoam': '#B8E2D4',
    'teal': '#A6DDE8',
    'sky': '#C5D9F2',
    'lavender': '#D6C7EE',
    'ink': '#4E5A67',
}

word_colors = {}
colors_list = ['#C8E6C9', '#B8E2D4', '#A6DDE8', '#C5D9F2', '#D6C7EE', '#9FD8F7', '#B5EAD7', '#FFDAB9', '#FFD4B4', '#E2C7EE', '#D4E6F1', '#F0E2C3', '#E8D5B7', '#C9E4CA']
for i, word in enumerate(filler_words):
    word_colors[word] = colors_list[i % len(colors_list)]

print(f'Loaded {len(records)} recordings with {len(filler_words)} filler words: {", ".join(filler_words)}')

Loaded 45 recordings with 16 filler words: I guess, I mean, actually, basically, i guess, i mean, kind of, like, okay, right, so, uh, um, well, yeah, you know


In [69]:
# Plot 1: Interactive filler-word percentages per recording.
import plotly.graph_objects as go

if not records:
    print('No recordings available to plot.')
else:
    dropdown_options = [
        ('Last 5 recordings', 5),
        ('Last 10 recordings', 10),
        ('Last 15 recordings', 15),
        ('All recordings', None),
    ]

    def build_view(limit):
        recency_sorted = sorted(records, key=lambda r: r['recording_dt'], reverse=True)
        selected = recency_sorted[:limit] if limit else recency_sorted
        selected = sorted(selected, key=lambda r: r['filler_percentage'], reverse=True)

        labels = [r['recording_label'] for r in selected]
        dates = [r['recording_dt'].strftime('%b %d, %Y') for r in selected]
        totals = np.array([r['filler_percentage'] for r in selected], dtype=float)

        x_by_word, custom_by_word, has_data_by_word = [], [], []
        for word in filler_words:
            values = np.array(
                [100.0 * r['filler_counts'].get(word, 0) / max(r['total_words'], 1)
                 for r in selected], dtype=float)
            custom = np.column_stack([
                np.array([word] * len(selected), dtype=object),
                np.array(labels, dtype=object),
                np.array(dates, dtype=object),
                totals,
                np.array([r['total_words'] for r in selected], dtype=float),
            ])
            x_by_word.append(values)
            custom_by_word.append(custom)
            has_data_by_word.append(bool(np.any(values > 0)))

        return {'labels': labels, 'x_by_word': x_by_word, 'custom_by_word': custom_by_word,
                'has_data_by_word': has_data_by_word}

    views = {name: build_view(limit) for name, limit in dropdown_options}
    default_view_name = 'All recordings'
    default_view = views[default_view_name]

    max_labels = max(len(v['labels']) for v in views.values())
    height_calc = max(450, min(1000, 110 + max_labels * 36))
    max_filler = max((r['filler_percentage'] for r in records), default=10.0)
    x_ceiling = min(100.0, max_filler * 1.2 + 3)

    fig = go.Figure()
    for idx, word in enumerate(filler_words):
        fig.add_trace(go.Bar(
            y=default_view['labels'],
            x=default_view['x_by_word'][idx],
            name=word,
            orientation='h',
            showlegend=default_view['has_data_by_word'][idx],
            marker_color=word_colors[word],
            customdata=default_view['custom_by_word'][idx],
            hovertemplate=(
                '<b>%{customdata[1]}</b><br>'
                'Date: %{customdata[2]}<br>'
                'Filler word: %{customdata[0]}<br>'
                'Word share: %{x:.2f}%<br>'
                'Total filler: %{customdata[3]:.2f}%<br>'
                'Total words: %{customdata[4]:.0f}<extra></extra>'
            ),
        ))

    dropdown_buttons = []
    for name, _ in dropdown_options:
        view = views[name]
        dropdown_buttons.append({
            'label': name,
            'method': 'update',
            'args': [
                {'x': view['x_by_word'],
                 'y': [view['labels']] * len(filler_words),
                 'customdata': view['custom_by_word'],
                 'showlegend': view['has_data_by_word']},
                {'title.text': f'Filler words as percentages per recording ({name})',
                 'yaxis.categoryarray': view['labels'],
                 'yaxis.categoryorder': 'array',
                 'yaxis.autorange': 'reversed',
                 'height': max(450, min(1000, 110 + len(view['labels']) * 36))},
            ],
        })

    fig.update_layout(
        barmode='stack',
        title=dict(text=f'Filler words as percentages per recording ({default_view_name})',
                   x=0.0, xanchor='left', font=dict(size=15)),
        xaxis=dict(range=[0, x_ceiling], ticksuffix='%',
                   title=dict(text='Percentage of all spoken words', standoff=12)),
        yaxis=dict(categoryarray=default_view['labels'], categoryorder='array', autorange='reversed'),
        template='plotly_white',
        height=height_calc,
        margin=dict(l=240, r=35, t=115, b=110),
        legend=dict(orientation='h', yanchor='top', y=-0.14, xanchor='left', x=0,
                    title_text='Filler word', font=dict(size=11)),
        updatemenus=[{
            'buttons': dropdown_buttons,
            'active': 3,
            'direction': 'down',
            'showactive': True,
            'x': 0.0, 'xanchor': 'left',
            'y': 1.0, 'yanchor': 'bottom',
            'pad': {'b': 8},
            'bgcolor': 'white', 'bordercolor': '#B7C7E5', 'borderwidth': 1,
        }],
    )
    fig.show()

In [70]:
# Plot 2: Filler words over time as interactive line chart with time frame selector.
import plotly.graph_objects as go
from dateutil.relativedelta import relativedelta
from datetime import timedelta

if not records:
    print('No recordings available to plot.')
else:
    time_ordered_records = sorted(records, key=lambda r: (r['recording_dt'], r['recording_id']))
    today_datetime = datetime.combine(TODAY, datetime.min.time())

    monday_this_week = today_datetime - timedelta(days=today_datetime.weekday())
    sunday_this_week = monday_this_week + timedelta(days=6)
    monday_last_week = monday_this_week - timedelta(days=7)
    sunday_last_week = monday_last_week + timedelta(days=6)

    time_frames = {
        'This week':  ('week',   monday_this_week,                         sunday_this_week),
        'Last week':  ('week',   monday_last_week,                         sunday_last_week),
        'Month':      ('month',  today_datetime.replace(day=1),            today_datetime),
        '6 months':   ('period', today_datetime - relativedelta(months=6), today_datetime),
        'Year':       ('period', today_datetime.replace(month=1, day=1),   today_datetime),
        'All time':   ('all',    None,                                      None),
    }

    XFMT = {
        'This week':  ('%a %d', 86400000),
        'Last week':  ('%a %d', 86400000),
        'Month':      ('%b %d', None),
        '6 months':   ('%b %Y', 'M1'),
        'Year':       ('%b',    'M1'),
        'All time':   ('%b %Y', 'M1'),
    }

    def get_filtered_data(frame_type, start_date, end_date):
        if frame_type == 'all':
            return time_ordered_records
        return [r for r in time_ordered_records if start_date <= r['recording_dt'] <= end_date]

    def xaxis_range(frame_type, start_date, end_date):
        if frame_type == 'all':
            if not time_ordered_records:
                return [None, None]
            dts = [r['recording_dt'] for r in time_ordered_records]
            pad = timedelta(days=20)
            return [(min(dts) - pad).isoformat(), (max(dts) + pad).isoformat()]
        pad = timedelta(days=1)
        return [(start_date - pad).isoformat(), (end_date + pad).isoformat()]

    NO_DATA_ANNOTATION = dict(
        text='No recordings in this timeframe',
        xref='paper', yref='paper',
        x=0.5, y=0.5,
        showarrow=False,
        font=dict(size=17, color='#aaaaaa'),
        xanchor='center', yanchor='middle',
    )

    frame_data = {}
    for frame_name, (frame_type, start_date, end_date) in time_frames.items():
        filtered = get_filtered_data(frame_type, start_date, end_date)
        traces = {}
        max_val = 0.0
        for word in filler_words:
            if filtered:
                dates  = [r['recording_dt'] for r in filtered]
                values = np.array(
                    [100.0 * r['filler_counts'].get(word, 0) / max(r['total_words'], 1)
                     for r in filtered], dtype=float)
                has_data = bool(np.any(values > 0))
                max_val  = max(max_val, float(np.nanmax(values)) if has_data else 0.0)
            else:
                dates, values, has_data = [], np.array([]), False
            traces[word] = {'dates': dates, 'values': values, 'has_data': has_data}
        frame_data[frame_name] = {
            'traces':  traces,
            'y_max':   max(5.0, max_val * 1.2),
            'empty':   len(filtered) == 0,
            'x_range': xaxis_range(frame_type, start_date, end_date),
        }

    default_frame = 'All time'
    default_data  = frame_data[default_frame]

    fig = go.Figure()
    for word in filler_words:
        td = default_data['traces'][word]
        fig.add_trace(go.Scatter(
            x=td['dates'], y=td['values'],
            name=word, mode='lines+markers',
            visible=td['has_data'],
            line=dict(color=word_colors[word], width=2.5),
            marker=dict(size=6),
            hovertemplate='<b>%{x|%b %d, %Y}</b><br>' + word + ': %{y:.2f}%<extra></extra>',
        ))

    timeframe_buttons = []
    for frame_name, (frame_type, start_date, end_date) in time_frames.items():
        fdata = frame_data[frame_name]
        fmt, dtick = XFMT[frame_name]
        new_x       = [fdata['traces'][w]['dates']    for w in filler_words]
        new_y       = [fdata['traces'][w]['values']   for w in filler_words]
        new_visible = [fdata['traces'][w]['has_data'] for w in filler_words]
        layout_update = {
            'title.text':       f'Filler words over time ({frame_name})',
            'xaxis.tickformat': fmt,
            'xaxis.range':      fdata['x_range'],
            'yaxis.range':      [0, fdata['y_max']],
        }
        if dtick is not None:
            layout_update['xaxis.dtick'] = dtick
        else:
            layout_update['xaxis.dtick'] = None
        layout_update['annotations'] = [NO_DATA_ANNOTATION] if fdata['empty'] else []
        timeframe_buttons.append({
            'label': frame_name, 'method': 'update',
            'args': [{'x': new_x, 'y': new_y, 'visible': new_visible}, layout_update],
        })

    default_fmt, default_dtick = XFMT[default_frame]

    fig.update_layout(
        title=dict(text=f'Filler words over time ({default_frame})',
                   x=0.0, xanchor='left', font=dict(size=15)),
        xaxis=dict(title='Recording date', tickformat=default_fmt,
                   dtick=default_dtick, range=default_data['x_range']),
        yaxis=dict(title='Percentage of spoken words (%)', range=[0, default_data['y_max']]),
        template='plotly_white', height=600, hovermode='closest',
        annotations=[] if not default_data['empty'] else [NO_DATA_ANNOTATION],
        legend=dict(orientation='h', yanchor='top', y=-0.14, xanchor='left', x=0, font=dict(size=11)),
        margin=dict(b=110, t=115, l=60, r=30),
        updatemenus=[{
            'buttons': timeframe_buttons, 'active': 5, 'direction': 'down', 'showactive': True,
            'x': 0.0, 'xanchor': 'left', 'y': 1.0, 'yanchor': 'bottom',
            'pad': {'b': 8}, 'bgcolor': 'white', 'bordercolor': '#B7C7E5', 'borderwidth': 1,
        }],
    )
    fig.show()

In [71]:
# Plot 3: Interactive pitch variation over time with time frame selector.
import plotly.graph_objects as go
from dateutil.relativedelta import relativedelta
from datetime import timedelta

if not records:
    print('No recordings available to plot.')
else:
    pitch_records = [record for record in records if np.isfinite(record['pitch_avg_variation'])]
    if not pitch_records:
        print('No voiced pitch values available to plot.')
    else:
        # Use TODAY as reference for all time frame calculations
        today_datetime = datetime.combine(TODAY, datetime.min.time())
        
        # Calculate Monday and Sunday of this week
        monday_this_week = today_datetime - timedelta(days=today_datetime.weekday())
        sunday_this_week = monday_this_week + timedelta(days=6)
        
        # Calculate Monday and Sunday of last week
        monday_last_week = monday_this_week - timedelta(days=7)
        sunday_last_week = monday_last_week + timedelta(days=6)
        
        # Define time frame filters with start and end dates
        time_frames = {
            'This week': ('week', monday_this_week, sunday_this_week),
            'Last week': ('week', monday_last_week, sunday_last_week),
            'Month': ('month', today_datetime.replace(day=1), today_datetime),
            '6 months': ('period', today_datetime - relativedelta(months=6), today_datetime),
            'Year': ('period', today_datetime.replace(month=1, day=1), today_datetime),
            'All time': ('all', None, None),
        }
        
        def get_filtered_pitch_data(frame_type, start_date, end_date):
            """Filter records based on date range"""
            if frame_type == 'all':
                return pitch_records
            if start_date is None:
                return pitch_records
            return [r for r in pitch_records if start_date <= r['recording_dt'] <= end_date]
        
        # Build data for each time frame
        frame_data = {}
        target_low, target_high = 3.0, 5.0
        
        for frame_name, (frame_type, start_date, end_date) in time_frames.items():
            filtered_records = get_filtered_pitch_data(frame_type, start_date, end_date)
            
            if not filtered_records:
                # Store empty state
                frame_data[frame_name] = {
                    'empty': True,
                    'message': f"You have not done any recordings in this chosen timeframe: '{frame_name}'",
                    # upper_limit is used symmetrically as [-upper_limit, upper_limit]
                    'upper_limit': target_high + 2.0,
                    'frame_type': frame_type,
                }
            else:
                dates = [r['recording_dt'] for r in filtered_records]
                avg_variation = np.array([r['pitch_avg_variation'] for r in filtered_records], dtype=float)
                min_variation = np.array([r['pitch_min_variation'] for r in filtered_records], dtype=float)
                max_variation = np.array([r['pitch_max_variation'] for r in filtered_records], dtype=float)
                overall_average = float(np.nanmean(avg_variation))
                # Make upper_limit symmetric based on absolute extremes
                upper_limit = max(target_high + 2.0, float(np.nanmax(np.abs(np.concatenate([max_variation, min_variation])))) * 1.15 if max_variation.size else target_high + 2.0)
                
                frame_data[frame_name] = {
                    'dates': dates,
                    'avg_variation': avg_variation,
                    'min_variation': min_variation,
                    'max_variation': max_variation,
                    'overall_average': overall_average,
                    'upper_limit': upper_limit,
                    'empty': False,
                    'frame_type': frame_type,
                }
        
        # Create figure with first time frame
        default_frame = 'All time'
        default_data = frame_data[default_frame]
        
        # Choose hover date format based on frame type
        hover_date_fmt = '%b %d, %Y'
        if default_data['frame_type'] == 'week':
            hover_date_fmt = '%a, %b %d'
        
        fig = go.Figure()

        if not default_data['empty']:
            dates = default_data['dates']
            # Polygon x coords used for filled traces
            poly_x = dates + dates[::-1]
            # Recommended positive band polygon (top then bottom reversed)
            pos_poly_y = [target_high for _ in dates] + [target_low for _ in dates[::-1]]
            # Recommended negative band polygon
            neg_poly_y = [-target_low for _ in dates] + [-target_high for _ in dates[::-1]]

            # Recommended band (positive) - visible in legend
            fig.add_trace(
                go.Scatter(
                    x=poly_x,
                    y=pos_poly_y,
                    fill='toself',
                    fillcolor=COLORS['sky'],
                    opacity=0.25,
                    line=dict(color='rgba(0,0,0,0)'),
                    hoverinfo='skip',
                    name='Recommended band',
                    showlegend=True,
                )
            )
            # Recommended band (negative) - same group, don't duplicate legend entry
            fig.add_trace(
                go.Scatter(
                    x=poly_x,
                    y=neg_poly_y,
                    fill='toself',
                    fillcolor=COLORS['sky'],
                    opacity=0.25,
                    line=dict(color='rgba(0,0,0,0)'),
                    hoverinfo='skip',
                    name='Recommended band',
                    showlegend=False,
                )
            )

            # Min-max band
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'] + default_data['dates'][::-1],
                    y=list(default_data['max_variation']) + list(default_data['min_variation'][::-1]),
                    fill='toself',
                    fillcolor=COLORS['seafoam'],
                    opacity=0.38,
                    line_color='rgba(0,0,0,0)',
                    hoverinfo='skip',
                    name='Min-max band',
                    showlegend=True,
                )
            )

            # Average variation line (include min/max in hover via customdata)
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'],
                    y=default_data['avg_variation'],
                    name='Average variation',
                    mode='lines+markers',
                    line=dict(color=COLORS['teal'], width=2.5),
                    marker=dict(size=6),
                    customdata=np.column_stack([default_data['min_variation'], default_data['max_variation']]),
                    hovertemplate='<b>%{x|' + hover_date_fmt + '}</b><br>Avg: %{y:.2f} st<br>Min: %{customdata[0]:.2f} st<br>Max: %{customdata[1]:.2f} st<extra></extra>',
                )
            )

            # Overall average line
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'],
                    y=np.full_like(default_data['avg_variation'], default_data['overall_average']),
                    name=f'Overall average ({default_data["overall_average"]:.2f} st)',
                    mode='lines',
                    line=dict(color=COLORS['ink'], width=2, dash='dash'),
                    hovertemplate='<b>%{x|' + hover_date_fmt + '}</b><br>Overall: ' + f'{default_data["overall_average"]:.2f}' + ' st<extra></extra>',
                )
            )

            # Zero reference line (keep as shape)
            fig.add_shape(
                type='line', xref='paper', x0=0, x1=1, yref='y', y0=0, y1=0,
                line=dict(color='rgba(0,0,0,0.35)', dash='dash'),
            )
        else:
            # Add empty annotation
            fig.add_annotation(
                text=default_data['message'],
                xref='paper',
                yref='paper',
                x=0.5,
                y=0.5,
                showarrow=False,
                font=dict(size=16, color='#999999'),
            )
        
        # Create time frame dropdown buttons
        timeframe_buttons = []
        for frame_name, (frame_type, start_date, end_date) in time_frames.items():
            fdata = frame_data[frame_name]
            
            # Determine hover date format for this frame
            if frame_type == 'week':
                hover_fmt = '%a, %b %d'
            else:
                hover_fmt = '%b %d, %Y'
            
            if fdata['empty']:
                # Empty state button: 5 traces (recommended pos, recommended neg, minmax, avg, overall)
                timeframe_buttons.append(
                    {
                        'label': frame_name,
                        'method': 'update',
                        'args': [
                            {
                                'x': [[], [], [], [], []],
                                'y': [[], [], [], [], []],
                            },
                            {
                                'title': f'Pitch variation over time ({frame_name})',
                                'annotations': [{
                                    'text': fdata['message'],
                                    'xref': 'paper',
                                    'yref': 'paper',
                                    'x': 0.5,
                                    'y': 0.5,
                                    'showarrow': False,
                                    'font': dict(size=16, color='#999999'),
                                }],
                                # symmetric y-range
                                'yaxis.range': [-fdata['upper_limit'], fdata['upper_limit']],
                            },
                        ],
                    }
                )
            else:
                # Build recommended polygons for this frame
                dates_f = fdata['dates']
                poly_x_f = dates_f + dates_f[::-1]
                pos_poly_y_f = [target_high for _ in dates_f] + [target_low for _ in dates_f[::-1]]
                neg_poly_y_f = [-target_low for _ in dates_f] + [-target_high for _ in dates_f[::-1]]

                timeframe_buttons.append(
                    {
                        'label': frame_name,
                        'method': 'update',
                        'args': [
                            {
                                'x': [poly_x_f, poly_x_f, fdata['dates'] + fdata['dates'][::-1], fdata['dates'], fdata['dates']],
                                'y': [pos_poly_y_f, neg_poly_y_f, list(fdata['max_variation']) + list(fdata['min_variation'][::-1]), fdata['avg_variation'], np.full_like(fdata['avg_variation'], fdata['overall_average'])],
                                'hovertemplate': [
                                    'skip',
                                    'skip',
                                    'skip',
                                    '<b>%{x|' + hover_fmt + '}</b><br>Avg: %{y:.2f} st<br>Min: %{customdata[0]:.2f} st<br>Max: %{customdata[1]:.2f} st<extra></extra>',
                                    '<b>%{x|' + hover_fmt + '}</b><br>Overall: ' + f'{fdata["overall_average"]:.2f}' + ' st<extra></extra>',
                                ],
                                'customdata': [None, None, None, np.column_stack([fdata['min_variation'], fdata['max_variation']]), None],
                            },
                            {
                                'title': f'Pitch variation over time ({frame_name})',
                                'annotations': [],
                                'yaxis.range': [-fdata['upper_limit'], fdata['upper_limit']],
                            },
                        ],
                    }
                )

        fig.update_layout(
            title=f'Pitch variation over time ({default_frame})',
            xaxis_title='Session date',
            yaxis_title='Pitch variation (semitones from session median)',
            template='plotly_white',
            height=600,
            hovermode='x unified',
            # symmetric y-axis centered on zero
            yaxis=dict(range=[-default_data['upper_limit'], default_data['upper_limit']]),
            legend=dict(
                orientation='h',
                yanchor='top',
                y=-0.22,
                xanchor='center',
                x=0.5,
                font=dict(size=11),
            ),
            margin=dict(b=160, t=100),
            updatemenus=[
                {
                    'buttons': timeframe_buttons,
                    'active': 5,
                    'direction': 'down',
                    'showactive': True,
                    'x': 0.0,
                    'xanchor': 'left',
                    'y': 1.08,
                    'yanchor': 'top',
                    'bgcolor': 'white',
                    'bordercolor': '#B7C7E5',
                    'borderwidth': 1,
                }
            ],
        )

        fig.show()


In [73]:
# Plot 4: Interactive WPM over time with time frame selector.
import plotly.graph_objects as go
from dateutil.relativedelta import relativedelta
from datetime import timedelta

if not records:
    print('No recordings available to plot.')
else:
    wpm_records = [r for r in records if np.isfinite(r['wpm'])]
    if not wpm_records:
        print('No WPM values available to plot.')
    else:
        today_datetime = datetime.combine(TODAY, datetime.min.time())

        monday_this_week = today_datetime - timedelta(days=today_datetime.weekday())
        sunday_this_week = monday_this_week + timedelta(days=6)
        monday_last_week = monday_this_week - timedelta(days=7)
        sunday_last_week = monday_last_week + timedelta(days=6)

        time_frames = {
            'This week':  ('week',   monday_this_week,                         sunday_this_week),
            'Last week':  ('week',   monday_last_week,                         sunday_last_week),
            'Month':      ('month',  today_datetime.replace(day=1),            today_datetime),
            '6 months':   ('period', today_datetime - relativedelta(months=6), today_datetime),
            'Year':       ('period', today_datetime.replace(month=1, day=1),   today_datetime),
            'All time':   ('all',    None,                                      None),
        }

        XFMT = {
            'This week':  ('%a %d', 86400000),
            'Last week':  ('%a %d', 86400000),
            'Month':      ('%b %d', None),
            '6 months':   ('%b %Y', 'M1'),
            'Year':       ('%b',    'M1'),
            'All time':   ('%b %Y', 'M1'),
        }

        target_low, target_high = 130.0, 150.0

        def get_filtered_wpm(frame_type, start_date, end_date):
            if frame_type == 'all':
                return wpm_records
            return [r for r in wpm_records if start_date <= r['recording_dt'] <= end_date]

        def xaxis_range(frame_type, start_date, end_date):
            if frame_type == 'all':
                dts = [r['recording_dt'] for r in wpm_records]
                pad = timedelta(days=20)
                return [(min(dts) - pad).isoformat(), (max(dts) + pad).isoformat()]
            pad = timedelta(days=1)
            return [(start_date - pad).isoformat(), (end_date + pad).isoformat()]

        NO_DATA_ANNOTATION = dict(
            text='No recordings in this timeframe',
            xref='paper', yref='paper',
            x=0.5, y=0.5,
            showarrow=False,
            font=dict(size=17, color='#aaaaaa'),
            xanchor='center', yanchor='middle',
        )

        frame_data = {}
        for frame_name, (frame_type, start_date, end_date) in time_frames.items():
            filtered = get_filtered_wpm(frame_type, start_date, end_date)
            if not filtered:
                frame_data[frame_name] = {
                    'empty': True, 'y_max': 200.0,
                    'x_range': xaxis_range(frame_type, start_date, end_date),
                }
            else:
                dates = [r['recording_dt'] for r in filtered]
                wpm_v = np.array([r['wpm'] for r in filtered], dtype=float)
                labels = [r['recording_label'] for r in filtered]
                if len(wpm_v) >= 2:
                    dn = mdates.date2num(dates)
                    trend = np.polyval(np.polyfit(dn, wpm_v, 1), dn)
                else:
                    trend = wpm_v.copy()
                frame_data[frame_name] = {
                    'empty': False,
                    'dates': dates,
                    'wpm_v': wpm_v,
                    'labels': labels,
                    'trend': trend,
                    'y_max': max(180.0, float(np.nanmax(wpm_v)) * 1.18),
                    'x_range': xaxis_range(frame_type, start_date, end_date),
                }

        default_frame = 'All time'
        dd = frame_data[default_frame]

        fig = go.Figure()

        # Recommended range: paper-coord shape so it always spans full width.
        fig.add_shape(
            type='rect',
            xref='paper', x0=0, x1=1,
            yref='y', y0=target_low, y1=target_high,
            fillcolor='rgba(197,217,242,0.25)',
            line_width=0,
        )

        if not dd['empty']:
            fig.add_trace(go.Scatter(
                x=dd['dates'],
                y=dd['wpm_v'],
                name='WPM',
                mode='markers',
                marker=dict(symbol='circle', size=8, color=COLORS['teal']),
                customdata=np.column_stack([dd['labels'], dd['wpm_v']]),
                hovertemplate='<b>%{customdata[0]}</b><br>Date: %{x|%b %d, %Y}<br>WPM: %{customdata[1]:.1f}<extra></extra>',
            ))
            fig.add_trace(go.Scatter(
                x=dd['dates'],
                y=dd['trend'],
                name='Trend line',
                mode='lines',
                line=dict(color=COLORS['lavender'], width=2.5, dash='dash'),
                hovertemplate='<b>%{x|%b %d, %Y}</b><br>Trend: %{y:.1f}<extra></extra>',
            ))
        else:
            fig.add_trace(go.Scatter(x=[], y=[], name='WPM', mode='markers'))
            fig.add_trace(go.Scatter(x=[], y=[], name='Trend line', mode='lines'))

        # Legend-only swatch for the recommended band.
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode='lines',
            line=dict(color=COLORS['sky'], width=3),
            name='Recommended band',
            hoverinfo='skip',
            showlegend=True,
        ))

        timeframe_buttons = []
        for frame_name, (frame_type, start_date, end_date) in time_frames.items():
            fdata = frame_data[frame_name]
            fmt, dtick = XFMT[frame_name]
            if fdata['empty']:
                trace_update = {'x': [[], [], []], 'y': [[], [], []], 'name': ['WPM', 'Trend line', 'Recommended band']}
            else:
                trace_update = {
                    'x': [fdata['dates'], fdata['dates'], [None]],
                    'y': [fdata['wpm_v'], fdata['trend'], [None]],
                    'customdata': [np.column_stack([fdata['labels'], fdata['wpm_v']]), None, None],
                    'name': ['WPM', 'Trend line', 'Recommended band'],
                }
            layout_update = {
                'title.text': f'Words per minute over time ({frame_name})',
                'xaxis.tickformat': fmt,
                'xaxis.range': fdata['x_range'],
                'yaxis.range': [0, fdata['y_max']],
            }
            if dtick is not None:
                layout_update['xaxis.dtick'] = dtick
            else:
                layout_update['xaxis.dtick'] = None
            layout_update['annotations'] = [NO_DATA_ANNOTATION] if fdata['empty'] else []
            timeframe_buttons.append({
                'label': frame_name,
                'method': 'update',
                'args': [trace_update, layout_update],
            })

        default_fmt, default_dtick = XFMT[default_frame]

        fig.update_layout(
            title=dict(text=f'Words per minute over time ({default_frame})', x=0.0, xanchor='left', font=dict(size=15)),
            xaxis=dict(title='Session date', tickformat=default_fmt, dtick=default_dtick,
                       range=dd['x_range'] if not dd['empty'] else [None, None]),
            yaxis=dict(title='Words per minute (WPM)', range=[0, dd['y_max'] if not dd['empty'] else 200.0]),
            template='plotly_white',
            height=600,
            hovermode='closest',
            annotations=[] if not dd['empty'] else [NO_DATA_ANNOTATION],
            legend=dict(
                orientation='h',
                yanchor='top',
                y=-0.14,
                xanchor='left',
                x=0,
                font=dict(size=11),
                traceorder='normal',
            ),
            margin=dict(b=110, t=115, l=60, r=30),
            updatemenus=[{
                'buttons': timeframe_buttons,
                'active': 5,
                'direction': 'down',
                'showactive': True,
                'x': 0.0,
                'xanchor': 'left',
                'y': 1.0,
                'yanchor': 'bottom',
                'pad': {'b': 8},
                'bgcolor': 'white',
                'bordercolor': '#B7C7E5',
                'borderwidth': 1,
            }],
        )
        fig.show()